In [0]:
!pip install boto3 wfdb matplotlib 

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import gc, shutil, os, ast, tempfile, threading, math
import pandas as pd
import numpy as np
import scipy.signal as signal
import wfdb
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from rich import print

# --- 1. CONFIGURACIÓN DE CREDENCIALES DIRECTAS ---
def get_s3_writer():
    """Usa las credenciales AWS explícitas para conectarse a S3."""
    try:
        print("[green]✓ Usando credenciales AWS explícitas proporcionadas...[/green]")
        return boto3.client(
            's3',
            aws_access_key_id='',
            aws_secret_access_key=''
            # Puedes agregar region_name='us-east-1' aquí si tu bucket principal lo requiere
        )
    except Exception as e:
        print(f"[red]Error al inicializar cliente boto3: {e}[/red]")
        raise e

# Cliente para leer de PhysioNet (Anónimo)
S3_READER = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))
# Cliente para escribir en tu bucket (Usa tus llaves directas)
S3_WRITER = get_s3_writer()

BUCKET_PHYSIONET = 'physionet-open'
BUCKET_OUTPUT = 'shazam-proyecto-ecg'
TARGET_FS = 250
TARGET_LEADS = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# Usamos /tmp (SSD local) para máxima velocidad y evitar errores de I/O de DBFS
OUTPUT_DIR_BASE = '/tmp/ecg_data'
OUTPUT_PDF_BASE = '/tmp/ejemplos_clase0'

MAX_WORKERS = 8
all_segments = []
metadata_list = []
_lock = threading.Lock()

# --- 2. PROCESAMIENTO Y FILTROS ---
INVALID_CODES = {'AFIB', 'AFLT', 'LBBB', 'RBBB', 'PVC', 'PAC', 'STTC', 'HYP', 'MI', 'CD', 'PACE'}

def is_clean_normal(scp_dict):
    if 'NORM' not in scp_dict: return False
    for code in scp_dict.keys():
        if code in INVALID_CODES: return False
    return True

def process_ecg_signal(raw_signal, fs):
    if fs != TARGET_FS:
        factor = math.gcd(int(TARGET_FS), int(fs))
        raw_signal = signal.resample_poly(raw_signal, int(TARGET_FS // factor), int(fs // factor))
    nyq = 0.5 * TARGET_FS
    b, a = signal.butter(5, [0.5 / nyq, 45.0 / nyq], btype='band')
    filtered = signal.filtfilt(b, a, raw_signal)
    return (filtered - np.mean(filtered)) / (np.std(filtered) + 1e-8)

def filter_quality(ventanas):
    if len(ventanas) == 0: return ventanas
    validos = [v for v in ventanas if not (np.isnan(v).any() or np.isinf(v).any() or np.any(np.std(v, axis=1) < 1e-4) or np.max(np.abs(v)) > 20.0)]
    return np.array(validos)

# --- 3. EXTRACCIÓN ---
def extract_centered_beats(clean_signals, fs_original, annotations, window_samples):
    n_leads, length = clean_signals.shape
    rms_signal = np.sqrt(np.mean(clean_signals**2, axis=0))
    if annotations is not None:
        original_peaks = [annotations.sample[i] for i, sym in enumerate(annotations.symbol) if sym == 'N']
        base_peaks = [int(p * (TARGET_FS / fs_original)) for p in original_peaks]
        peaks = [p - 50 + np.argmax(rms_signal[max(0, p-50):min(length, p+50)]) for p in base_peaks]
    else:
        peaks, _ = signal.find_peaks(rms_signal, distance=int(TARGET_FS * 0.45), prominence=np.std(rms_signal)*0.6)
    
    back, forward = int(window_samples * 0.35), window_samples - int(window_samples * 0.35)
    ventanas = [clean_signals[:, p-back:p+forward] for p in peaks if p-back >= 0 and p+forward <= length]
    bpm = 60 / (np.mean(np.diff(peaks)) / TARGET_FS) if len(peaks) > 1 else 0
    return np.array([v for v in ventanas if v.shape[1] == window_samples]), bpm

# --- 4. PARALELISMO ---
def ingest_record(path, subject, dataset_name, patient_id, split_name, use_annotations, cfg):
    base_name = subject.split('/')[-1]
    tmp_dir = tempfile.mkdtemp(dir='/tmp')
    local_base = os.path.join(tmp_dir, base_name)
    try:
        for ext in ['.hea', '.dat']: S3_READER.download_file(BUCKET_PHYSIONET, f"{path}{ext}", f"{local_base}{ext}")
        ann = None
        if use_annotations:
            try:
                S3_READER.download_file(BUCKET_PHYSIONET, f"{path}.atr", f"{local_base}.atr")
                ann = wfdb.rdann(local_base, 'atr')
            except: pass
        record = wfdb.rdrecord(local_base)
        sig_names = [s.upper() for s in record.sig_name]
        indices = [sig_names.index(ld) if ld in sig_names else sig_names.index('MLII') if ld=='II' and 'MLII' in sig_names else -1 for ld in TARGET_LEADS]
        if -1 not in indices:
            clean_sigs = np.array([process_ecg_signal(record.p_signal[:, idx], record.fs) for idx in indices])
            ventanas, bpm = extract_centered_beats(clean_sigs, record.fs, ann, cfg['muestras'])
            ventanas = filter_quality(ventanas)
            if len(ventanas) > 0:
                if len(ventanas) > cfg['max_per_patient']: ventanas = ventanas[np.random.choice(len(ventanas), cfg['max_per_patient'], replace=False)]
                with _lock:
                    all_segments.append(ventanas)
                    metadata_list.append({'dataset': dataset_name, 'patient_id': patient_id, 'split': split_name, 'registro': base_name, 'num_latidos': len(ventanas), 'bpm': bpm})
    except: pass
    finally: shutil.rmtree(tmp_dir, ignore_errors=True)

def run_parallel(tasks, description):
    print(f"[yellow]{description}...[/yellow]")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for f in as_completed([executor.submit(*t) for t in tasks]): f.result()

# --- 5. PDF CON GRILLA ECG ROJA (50 EJEMPLOS) ---
def generate_pdf_pro(all_segments, metadata_list, output_path, cfg):
    if not metadata_list: return
    num_muestras = cfg['muestras']
    back = int(num_muestras * 0.35)
    t = np.linspace(-back / TARGET_FS, (num_muestras - back) / TARGET_FS, num_muestras, endpoint=False)
    with PdfPages(output_path) as pdf:
        total_ejemplos = min(50, len(metadata_list))
        idx_pats = np.random.choice(len(metadata_list), total_ejemplos, replace=False)
        for i, idx in enumerate(idx_pats):
            latido = all_segments[idx][np.random.randint(len(all_segments[idx]))]
            fig, axs = plt.subplots(4, 3, figsize=(18, 16), sharex=True)
            fig.suptitle(f"Clase 0 (Normal) | REG: {metadata_list[idx]['registro']} | Ejemplo {i+1}/{total_ejemplos}", fontsize=18, fontweight='bold', y=0.96)
            axs = axs.flatten()
            for ch in range(12):
                ax = axs[ch]
                ax.plot(t, latido[ch], color='#1f77b4', linewidth=1.2)
                # GRILLA ROJA ECG REQUERIDA
                ax.grid(True, which='both', color='red', linestyle='--', linewidth=0.5, alpha=0.3)
                ax.minorticks_on()
                ax.grid(True, which='minor', color='red', linestyle=':', linewidth=0.2, alpha=0.2)
                ax.set_title(f"Derivación {TARGET_LEADS[ch]}", fontsize=12)
                ax.axvline(0, color='black', linewidth=1)
            plt.tight_layout(rect=[0, 0.03, 1, 0.93])
            pdf.savefig(fig)
            plt.close()

def export_data(all_segments, metadata_list, base_dir, cfg):
    chunk_size = 3000000 // cfg['muestras']
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(base_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        idx_split = [i for i, m in enumerate(metadata_list) if m['split'] == split]
        if not idx_split: continue
        buf, chunk, meta_final = [], 0, []
        for i in idx_split:
            for b_idx, beat in enumerate(all_segments[i]):
                buf.append(beat)
                meta_final.append({**metadata_list[i], 'beat_idx': b_idx, 'chunk': chunk})
                if len(buf) >= chunk_size:
                    np.savez_compressed(os.path.join(split_dir, f"normal_{chunk:03d}.npz"), signals=np.array(buf, dtype=np.float32))
                    buf, chunk = [], chunk + 1
        if buf: np.savez_compressed(os.path.join(split_dir, f"normal_{chunk:03d}.npz"), signals=np.array(buf, dtype=np.float32))
        pd.DataFrame(meta_final).to_csv(os.path.join(split_dir, "normal_meta.csv"), index=False)

# --- 6. FLUJO PRINCIPAL ---
S3_READER.download_file(BUCKET_PHYSIONET, 'ptb-xl/1.0.3/ptbxl_database.csv', 'ptbxl_database.csv')
df_ptb = pd.read_csv('ptbxl_database.csv')
df_ptb['scp_codes'] = df_ptb['scp_codes'].apply(ast.literal_eval)
df_sanos = df_ptb[df_ptb['scp_codes'].apply(is_clean_normal)].copy()
pts = df_sanos['patient_id'].unique()
np.random.shuffle(pts)
train_p, val_p = set(pts[:int(0.8*len(pts))]), set(pts[int(0.8*len(pts)):int(0.9*len(pts))])
get_sp = lambda p: 'train' if p in train_p else ('val' if p in val_p else 'test')

inc_recs = [f"I{str(i).zfill(2)}" for i in range(1, 76)]
np.random.shuffle(inc_recs)
train_i, val_i = set(inc_recs[:int(0.8*len(inc_recs))]), set(inc_recs[int(0.8*len(inc_recs)):int(0.9*len(inc_recs))])
get_si = lambda r: 'train' if r in train_i else ('val' if r in val_i else 'test')

configs = [
    {"name": "200", "muestras": 200, "max_per_patient": 400, "prefix": "silver_12leads/clase_0_normal/"},
    {"name": "500", "muestras": 500, "max_per_patient": 150, "prefix": "silver_12leads/clase_0_normal_500/"},
    {"name": "2000", "muestras": 2000, "max_per_patient": 50, "prefix": "silver_12leads/clase_0_normal_2000/"}
]

for cfg in configs:
    all_segments, metadata_list = [], []
    out_local, pdf_local = f"{OUTPUT_DIR_BASE}_{cfg['name']}", f"{OUTPUT_PDF_BASE}_{cfg['name']}.pdf"
    
    print(f"\n[bold magenta]>>> INICIANDO {cfg['name']} MUESTRAS <<<[/bold magenta]")
    tasks_p = [(ingest_record, f"ptb-xl/1.0.3/{r['filename_hr']}", r['filename_hr'], "PTB-XL", r['patient_id'], get_sp(r['patient_id']), False, cfg) for _, r in df_sanos.iterrows()]
    run_parallel(tasks_p, "Procesando PTB-XL")
    tasks_i = [(ingest_record, f"incartdb/1.0.0/{r}", r, "INCART", r, get_si(r), True, cfg) for r in inc_recs]
    run_parallel(tasks_i, "Procesando INCART")

    if all_segments:
        export_data(all_segments, metadata_list, out_local, cfg)
        generate_pdf_pro(all_segments, metadata_list, pdf_local, cfg)
        
        print(f"\n[bold yellow]>>> Subiendo a S3 usando llaves directas... <<<[/bold yellow]")
        for s in ['train', 'val', 'test']:
            p_s = os.path.join(out_local, s)
            if os.path.exists(p_s):
                for f in os.listdir(p_s):
                    S3_WRITER.upload_file(os.path.join(p_s, f), BUCKET_OUTPUT, f"{cfg['prefix']}{s}/{f}")
        
        if os.path.exists(pdf_local): S3_WRITER.upload_file(pdf_local, BUCKET_OUTPUT, f"{cfg['prefix']}ejemplos_{cfg['name']}.pdf")
        print(f"[bold green]✓ {cfg['name']} completado.[/bold green]")

    shutil.rmtree(out_local, ignore_errors=True)
    if os.path.exists(pdf_local): os.remove(pdf_local)

print("[bold cyan]--- PROCESO FINALIZADO CON ÉXITO ---[/bold cyan]")

✓ Usando credenciales AWS explícitas proporcionadas...

>>> INICIANDO 200 MUESTRAS <<<

Procesando PTB-XL...

Procesando INCART...

>>> Subiendo a S3 usando llaves directas... <<<

...

Procesando PTB-XL...

Procesando INCART...

>>> Subiendo a S3 usando llaves directas... <<<

✓ 2000 completado.

--- PROCESO FINALIZADO CON ÉXITO ---